<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_LinearRegression/blob/main/Internship_LinearRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("properties_features.csv",  parse_dates=["listed_date"])

In [3]:
print( f"loaded {df.shape[0]} rows x {df.shape[1]} columns")

loaded 3000 rows x 71 columns


In [4]:
FORBIDDEN = [ "price_per_sqft", "rental_yield_pct",
             "log_price_aed",  "area_price_encoded",
              "days_on_market",    "sold_fast", "listing_status",
              "annual_rent_aed",    "log_annual_rent_aed"]

In [6]:
print("\nexcluded because they would leak the answer:")
for col in FORBIDDEN:
  print(f" {col}")


excluded because they would leak the answer:
 price_per_sqft
 rental_yield_pct
 log_price_aed
 area_price_encoded
 days_on_market
 sold_fast
 listing_status
 annual_rent_aed
 log_annual_rent_aed


In [7]:
model_df = df[df["price_aed"].notna()].copy()

In [8]:
print(f"\nmodelling population: {len(model_df)} listings")


modelling population: 3000 listings


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
DROP = FORBIDDEN + ["listed_date", "description", "price_aed","latitude", "longitude"]
X = model_df.drop(columns=[c for c in DROP if c in model_df.columns])
y_aed = model_df["price_aed"]
y = np.log1p(y_aed)
print(f"\nfeature matrix: {X.shape[1]} columns")


feature matrix: 57 columns


In [11]:
print( f"target: log1p(price_aed)," f"skew {y_aed.skew():.2f} -> {y.skew():.2f}")

target: log1p(price_aed),skew 6.76 -> 0.51


In [12]:
X_train, X_test, y_train, y_test, aed_train, aed_test = train_test_split(  X, y, y_aed, test_size=0.2, random_state=42)

In [13]:
print(f"train {len(X_train)}   test {len(X_test)}")

train 2400   test 600


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [15]:
numeric_cols = X.select_dtypes( include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(    include=["object"]).columns.tolist()

print(f"\nnumeric: {len(numeric_cols)}   categorical: {len(categorical_cols)}")
print(f"categorical columns: {categorical_cols}")


numeric: 47   categorical: 10
categorical columns: ['emirate', 'area', 'property_type', 'completion_status', 'developer', 'ownership_type', 'furnishing', 'view', 'listed_by', 'agency_tier']


In [16]:
preprocessor = ColumnTransformer([("num", Pipeline([ ("impute", SimpleImputer(strategy="median")),
                                                    ("scale", StandardScaler())
                                                  ]), numeric_cols),
                                ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                                  ("encode", OneHotEncoder( handle_unknown="ignore", sparse_output=False,
                                                  min_frequency=10 ))
                                                ]), categorical_cols)
                                ])

In [17]:
from sklearn.linear_model import LinearRegression

In [18]:
model = Pipeline([("prep", preprocessor),("lm", LinearRegression())])
model.fit(X_train, y_train)
n_features = model.named_steps["prep"].transform(X_train).shape[1]

In [19]:
print(f"\nfitted on {len(X_train)} rows x {n_features} features after encoding")


fitted on 2400 rows x 188 features after encoding


In [20]:
pred_log = model.predict(X_test)
pred_aed = np.expm1(pred_log)

print("\n--- raw model output, before converting back ---")
print(f"{np.round(pred_log[:4], 2)}   <- these are LOGS, not prices")


--- raw model output, before converting back ---
[13.91 12.92 14.58 14.88]   <- these are LOGS, not prices


In [21]:
comparison = pd.DataFrame({ "actual_aed": aed_test.values,
                           "predicted_aed": pred_aed,}).assign( error_aed=lambda d: d["predicted_aed"] - d["actual_aed"],
                                                               error_pct=lambda d: 100 * (d["predicted_aed"] - d["actual_aed"]) / d["actual_aed"])

In [22]:
print("\n--- five test predictions, in dirhams ---")
print(comparison.head(5).round(0).to_string(index=False))


--- five test predictions, in dirhams ---
 actual_aed  predicted_aed  error_aed  error_pct
  1000000.0      1100715.0   100715.0       10.0
   400000.0       408367.0     8367.0        2.0
  1950000.0      2137537.0   187537.0       10.0
  3200000.0      2910028.0  -289972.0       -9.0
  2850000.0      2752667.0   -97333.0       -3.0
